# Sudoku Solver

Alex Keyken 13/11/26

Simple sudoku solver that fills cells where only one digit is possible and recursively guesses otherwise. Code was written without use of AI.

In [44]:
N_BOARD_ROWS = 9
N_BOARD_COLS = N_BOARD_ROWS
N_BOX_ROWS = 3
N_BOX_COLS = N_BOX_ROWS

BOXES_PER_ROW = N_BOARD_ROWS//N_BOX_ROWS
BOXES_PER_COL = N_BOARD_COLS//N_BOX_COLS

def print_board(board):
    """
    Visualise board with box divider lines.
    """
    for iline in range(len(board)):
        if iline % N_BOX_ROWS == 0 and iline != 0:
            print("- " * 11)
        for ichar in range(len(board[0])):
            if ichar % N_BOX_COLS == 0 and ichar != 0:
                print("|", end=" ")
            print(board[iline][ichar], end=" ")
        print()

def get_box_digits(board: list[list[int]], cell: tuple[int, int]) -> list[int]:
    """
    Retrieves a list containing all digits inside the box that the cell belongs to. 
    """
    box_digit_list = []
    row, col = cell
    box_row_offset, box_col_offset = row//N_BOX_ROWS, col//N_BOX_COLS
    for r in range(N_BOX_ROWS):
        for k in range(N_BOX_COLS):
            box_digit_list.append(board[N_BOX_ROWS*box_row_offset+r][N_BOX_COLS*box_col_offset+k])
    
    return box_digit_list

def compute_cell_candidates(board: list[list[int]], cell: tuple[int, int]) -> list[int]:
    """
    Computes candidates (digits that would not cause rule breaks) for a cell.
    Can be improved as several checks overlap.
    """
    cell_candidate_list = []
    row, col = cell
    for digit in range(1, N_BOARD_ROWS+1):
        is_in_row = digit in board[row]
        is_in_col = digit in [board[row_index][col] for row_index in range(len(board))]
        is_in_box = digit in get_box_digits(board, cell)

        if not (is_in_row or is_in_col or is_in_box):
            cell_candidate_list.append(digit)

    return cell_candidate_list

def compute_board_candidates(board: list[list[int]]) -> dict[tuple[int, int], list[int]]:
    """
    Creates a dictionary containing a list of candidates per cell on the board.
    Indexed by the cell coordinates such that: dict[(row,col)] = [list of digits].
    """
    candidate_dict = {}
    for row in range(len(board)):
        for col in range(len(board[row])):
            if board[row][col] == 0:
                cell = (row, col)
                candidate_dict[cell] = compute_cell_candidates(board, cell)
                
    return candidate_dict

def fill_singletons(board: list[list[int]]) -> None:
    """
    Updates board in place by filling singletons (empty cells with only one candidate digit).
    Loops until no more singletons are found.
    """
    made_changes = True
    while made_changes:
        made_changes = False
        candidate_dict = compute_board_candidates(board)
        for cell, candidates in candidate_dict.items():
            if len(candidates) == 1:
                row, col = cell
                board[row][col] = candidates[0]
                made_changes = True
                
def solve(board: list[list[int]]) -> bool:
    """
    Solves board in place by filling singletons and guessing otherwise,
    recursively until board is full.
    """
    fill_singletons(board)
    
    candidates = compute_board_candidates(board)
    if not candidates:
        return check_solved_board(board)

    most_constrained_cell = min(candidates, key=lambda x: len(candidates[x]))
    
    row, col = most_constrained_cell
    for digit in candidates[most_constrained_cell]:
        trial = [r[:] for r in board]
        trial[row][col] = digit
        
        if solve(trial):
            board[:] = trial
            return True
            
    return False

def check_solved_board(board: list[list[int]]) -> bool:
    """
    Check completed board for repeated digits within each row, column and box.
    """
    
    for row in board:
        if len(row) != len(set(row)):
            return False

    transposed_board = list(map(list, zip(*board)))
    for col in transposed_board:
        if len(col) != len(set(col)):
            return False

    for box_row in range(BOXES_PER_ROW):
        for box_col in range(BOXES_PER_COL):
            box = get_box_digits(board, (N_BOX_ROWS*box_row, N_BOX_COLS*box_col))
            if len(box) != len(set(box)):
                return False     
                
    return True

In [67]:
test_sudoku = [
    [5,0,0,4,6,7,3,0,9],
    [9,0,3,8,1,0,4,2,7],
    [1,7,4,2,0,3,0,0,0],
    [2,3,1,9,7,6,8,5,4],
    [8,5,7,1,2,4,0,9,0],
    [4,9,6,3,0,8,1,7,2],
    [0,0,0,0,8,9,2,6,0],
    [7,8,2,6,4,1,0,0,5],
    [0,1,0,0,0,0,7,0,8]
]

test_sudoku_harder = [
    [5,0,0,4,6,7,0,0,9],
    [9,0,3,8,1,0,0,2,7],
    [1,7,4,2,0,3,0,0,0],
    [2,3,1,0,7,6,0,5,4],
    [8,5,7,1,2,4,0,9,0],
    [4,9,6,3,0,8,0,7,2],
    [0,0,0,0,8,9,0,6,0],
    [0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0]
]

5 0 0 | 4 6 7 | 3 0 9 
9 0 3 | 8 1 0 | 4 2 7 
1 7 4 | 2 0 3 | 0 0 0 
- - - - - - - - - - - 
2 3 1 | 9 7 6 | 8 5 4 
8 5 7 | 1 2 4 | 0 9 0 
4 9 6 | 3 0 8 | 1 7 2 
- - - - - - - - - - - 
0 0 0 | 0 8 9 | 2 6 0 
7 8 2 | 6 4 1 | 0 0 5 
0 1 0 | 0 0 0 | 7 0 8 
